# Hierarchical Cell Type Annotation for Spatial Transcriptomics

This notebook demonstrates a **hierarchical annotation workflow** for cell type identification in Xenium spatial transcriptomics data. We use the human lymph node dataset as an example.

**Dataset**: 10x Xenium Human Lymph Node, 708K cells, 4,600 genes

## Why Hierarchical Annotation?

Reference-based annotation methods can struggle with spatial data because:
- **Marker dropout**: Sparse gene detection means key markers may be undetected
- **Panel limitations**: Only ~500 genes vs 20,000+ in scRNA-seq
- **Cross-lineage contamination**: Without lineage constraints, cells can be misassigned

The hierarchical approach constrains subtype assignment within biologically valid lineages using canonical markers.

## Note on Reproducibility

This notebook documents the **principles and approach** used for annotation. For downstream analysis, you can use the pre-annotated data directly:

```python
adata = sc.read_h5ad('../data/xenium_lymphnode.h5ad')
cell_types = adata.obs['cell_type']
```

## Workflow Overview

1. **Lineage assignment** using canonical markers (B, T, Myeloid, Stromal, NK)
2. **Subtype refinement** using marker rules within each lineage
3. **Validation** with markers, proportions, and spatial co-localization

---
## 1. Setup & Data Loading

In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.sparse import issparse
from scipy.spatial import cKDTree
from sklearn.neighbors import NearestNeighbors
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Helper function to get gene expression
def get_expression(adata, gene):
    """Get expression array for a gene (handles sparse matrices)."""
    if gene not in adata.var_names:
        return np.zeros(adata.n_obs)
    X = adata[:, gene].X
    if issparse(X):
        X = X.toarray().flatten()
    return np.asarray(X).flatten()

def get_expr_fraction(adata, mask, gene):
    """Get fraction of cells expressing a gene."""
    if gene not in adata.var_names:
        return 0.0
    X = adata[mask, gene].X
    if issparse(X):
        X = X.toarray()
    return float((X > 0).mean())

In [2]:
# Load raw Xenium data
# Using the clean annotated file as source for raw expression
# In practice, you would start from the raw cell_feature_matrix.h5ad
DATA_PATH = '../data/xenium_lymphnode.h5ad'
adata = sc.read_h5ad(DATA_PATH)

print(f'Total cells: {adata.n_obs:,}')
print(f'Total genes: {adata.n_vars:,}')
print(f'Spatial coordinates: {adata.obsm["spatial"].shape}')

# Remove any existing annotations to start fresh
if 'cell_type' in adata.obs.columns:
    print(f'\\nRemoving existing cell_type column for fresh annotation')
    del adata.obs['cell_type']

Total cells: 708,647
Total genes: 4,624
Spatial coordinates: (708647, 2)
\nRemoving existing cell_type column for fresh annotation


---
## 2. Lineage Assignment

The first step is to assign each cell to a major **lineage** using canonical markers. This constrains downstream subtype assignment to biologically valid options.

### Lineage Markers

| Lineage | Canonical Markers | Function |
|:--------|:------------------|:---------|
| B cell | MS4A1, CD79A, CD79B, CD19, PAX5 | B lymphocyte identity |
| T cell | CD3E, CD3D, CD3G, CD2, CD7 | T lymphocyte identity |
| Myeloid | CD68, CD14, LYZ, CSF1R, ITGAM | Myeloid/macrophage markers |
| Stromal | PECAM1, VWF, CDH5, COL1A1, CXCL12 | Vascular & fibroblast markers |
| NK | NCAM1, NKG7, GNLY, KLRD1, KLRB1 | Natural killer cell markers |

For each cell, we calculate a **lineage score** = fraction of markers detected (>0). Cells are assigned to the lineage with highest score, provided it exceeds a threshold.

In [3]:
# Define canonical lineage markers
LINEAGE_MARKERS = {
    'B_cell': ['MS4A1', 'CD79A', 'CD79B', 'CD19', 'PAX5'],
    'T_cell': ['CD3E', 'CD3D', 'CD3G', 'CD2', 'CD7'],
    'Myeloid': ['CD68', 'CD14', 'LYZ', 'CSF1R', 'ITGAM'],
    'Stromal': ['PECAM1', 'VWF', 'CDH5', 'COL1A1', 'COL3A1', 'CXCL12'],
    'NK': ['NCAM1', 'NKG7', 'GNLY', 'KLRD1', 'KLRB1'],
}

# Check which markers are in the panel
print('Marker availability in Xenium panel:')
for lineage, markers in LINEAGE_MARKERS.items():
    available = [m for m in markers if m in adata.var_names]
    missing = [m for m in markers if m not in adata.var_names]
    print(f'  {lineage}: {len(available)}/{len(markers)} markers')
    if missing:
        print(f'    Missing: {missing}')

Marker availability in Xenium panel:
  B_cell: 5/5 markers
  T_cell: 4/5 markers
    Missing: ['CD3D']
  Myeloid: 4/5 markers
    Missing: ['LYZ']
  Stromal: 3/6 markers
    Missing: ['VWF', 'COL1A1', 'COL3A1']
  NK: 3/5 markers
    Missing: ['NKG7', 'GNLY']


In [4]:
def compute_lineage_scores(adata, lineage_markers):
    """
    Compute lineage scores for each cell.
    
    Score = fraction of markers detected (expression > 0)
    """
    scores = {}
    
    for lineage, markers in lineage_markers.items():
        available = [m for m in markers if m in adata.var_names]
        if not available:
            scores[f'{lineage}_score'] = np.zeros(adata.n_obs)
            scores[f'{lineage}_positive'] = np.zeros(adata.n_obs, dtype=bool)
            continue
        
        # Score = fraction of markers detected
        detected = np.zeros(adata.n_obs)
        any_positive = np.zeros(adata.n_obs, dtype=bool)
        for gene in available:
            expr = get_expression(adata, gene)
            detected += (expr > 0).astype(float)
            any_positive |= (expr > 0)
        
        scores[f'{lineage}_score'] = detected / len(available)
        scores[f'{lineage}_positive'] = any_positive
    
    return pd.DataFrame(scores, index=adata.obs_names)

# Compute lineage scores
print('Computing lineage scores...')
lineage_scores = compute_lineage_scores(adata, LINEAGE_MARKERS)

# Store in adata
for col in lineage_scores.columns:
    adata.obs[col] = lineage_scores[col].values

# Summary statistics
print('\nLineage score statistics:')
for lineage in LINEAGE_MARKERS.keys():
    score_col = f'{lineage}_score'
    pos_col = f'{lineage}_positive'
    mean_score = adata.obs[score_col].mean()
    frac_pos = adata.obs[pos_col].mean()
    print(f'  {lineage}: mean={mean_score:.3f}, any_positive={100*frac_pos:.1f}%')

Computing lineage scores...



Lineage score statistics:
  B_cell: mean=0.196, any_positive=45.1%
  T_cell: mean=0.230, any_positive=57.6%
  Myeloid: mean=0.055, any_positive=17.0%
  Stromal: mean=0.164, any_positive=43.1%
  NK: mean=0.018, any_positive=5.1%


In [5]:
# Assign primary lineage
LINEAGE_THRESHOLD = 0.2  # At least 20% of markers detected

score_cols = [f'{lin}_score' for lin in LINEAGE_MARKERS.keys()]
lineage_df = adata.obs[score_cols]

# Get max score and corresponding lineage
adata.obs['lineage_score_max'] = lineage_df.max(axis=1)
adata.obs['primary_lineage'] = lineage_df.idxmax(axis=1).str.replace('_score', '')

# Mark low-confidence cells as Unassigned
low_conf = adata.obs['lineage_score_max'] < LINEAGE_THRESHOLD
adata.obs.loc[low_conf, 'primary_lineage'] = 'Unassigned'

# Distribution
print(f'Primary lineage distribution (threshold={LINEAGE_THRESHOLD}):')
lin_counts = adata.obs['primary_lineage'].value_counts()
for lin, n in lin_counts.items():
    pct = 100 * n / adata.n_obs
    print(f'  {lin:<12}: {n:>7,} ({pct:>5.1f}%)')

Primary lineage distribution (threshold=0.2):
  T_cell      : 253,048 ( 35.7%)
  B_cell      : 179,764 ( 25.4%)
  Stromal     : 174,603 ( 24.6%)
  Unassigned  :  62,091 (  8.8%)
  Myeloid     :  29,981 (  4.2%)
  NK          :   9,160 (  1.3%)


In [6]:
# Validate lineage assignment with marker expression
VALIDATION_MARKERS = {
    'B_cell': ['MS4A1', 'CD79A'],
    'T_cell': ['CD3E', 'CD3D'],
    'Myeloid': ['CD68', 'CD14'],
    'Stromal': ['PECAM1', 'CXCL12'],
    'NK': ['KLRB1', 'NKG7'],
}

print('Lineage validation (% cells positive for key markers):')
print(f'{"Lineage":<12} {"n_cells":>8}  {"Marker1":>10} {"Marker2":>10}')
print('-' * 50)

for lineage, markers in VALIDATION_MARKERS.items():
    mask = adata.obs['primary_lineage'] == lineage
    n = mask.sum()
    if n == 0:
        continue
    frac1 = get_expr_fraction(adata, mask, markers[0]) * 100 if markers[0] in adata.var_names else 0
    frac2 = get_expr_fraction(adata, mask, markers[1]) * 100 if len(markers) > 1 and markers[1] in adata.var_names else 0
    print(f'{lineage:<12} {n:>8,}  {frac1:>9.1f}% {frac2:>9.1f}%')

Lineage validation (% cells positive for key markers):
Lineage       n_cells     Marker1    Marker2
--------------------------------------------------


B_cell        179,764       85.2%      72.8%


T_cell        253,048       92.4%       0.0%
Myeloid        29,981       71.8%      47.3%


Stromal       174,603       37.0%      77.8%
NK              9,160       83.2%       0.0%


---
## 3. Subtype Refinement

Now we refine cell types within each lineage using marker-based rules. This hierarchical approach ensures subtypes are biologically valid.

### T Cell Subtypes

| Subtype | Markers | Description |
|:--------|:--------|:------------|
| T_CD8 | CD8A+ or CD8B+ | Cytotoxic T cells |
| T_helper | CD4+, CD8A- | Helper T cells |
| Treg | CD3E+, FOXP3+ | Regulatory T cells |
| Tfh | CD3E+, CXCR5+, FOXP3- | Follicular helper T |
| Tfr | CXCR5+, FOXP3+ | Follicular regulatory T |
| T_naive | CCR7+ or LEF1+ | Naive T cells |
| T_DN | CD3E+, CD4-, CD8A- | Double negative T |

### B Cell Subtypes

| Subtype | Markers | Description |
|:--------|:--------|:------------|
| GC_DZ | MKI67+ or TOP2A+ | GC dark zone (proliferating B cells) |
| GC_LZ | BCL6+, MKI67-, TOP2A- | GC light zone (non-proliferating) |
| NBC | FCER2+ or TCL1A+ | Naive B cell |
| MBC | CD27+ | Memory B cell |
| PC | XBP1 high, MS4A1 low | Plasma cell |
| PB | XBP1+, MS4A1+ | Plasmablast |

**Note on GC_DZ**: BCL6 detection is often low (~15%) in spatial transcriptomics due to marker dropout. Proliferation markers (MKI67, TOP2A) are more reliably detected and serve as the primary identifier for dark zone cells.

### Stromal Subtypes (including from Unassigned lineage)

| Subtype | Markers | Description |
|:--------|:--------|:------------|
| Endothelial | PECAM1+ or VWF+ | Vascular endothelium |
| Pericyte | RGS5+ or PDGFRB+ | Vascular pericytes |
| FRC | CXCL12+ or CCL19+ | Fibroblastic reticular cells |
| FDC | CXCL13+ | Follicular dendritic cells |

**Note on FRC/FDC**: Many FRC and FDC cells fall below the lineage marker threshold (assigned as "Unassigned"). These cells are recovered based on their characteristic chemokine expression: CXCL12+ (T-zone chemokine) for FRC and CXCL13+ (B-zone chemokine) for FDC.

In [7]:
# Subtype refinement within lineages
print('Refining subtypes within lineages...')

# Initialize with primary lineage
adata.obs['refined_subtype'] = adata.obs['primary_lineage'].copy()

# --- T cell subtypes ---
t_mask = adata.obs['primary_lineage'] == 'T_cell'
print(f'\nT cells: {t_mask.sum():,}')

cd8a = get_expression(adata, 'CD8A')
cd8b = get_expression(adata, 'CD8B') if 'CD8B' in adata.var_names else np.zeros(adata.n_obs)
cd4 = get_expression(adata, 'CD4')
cd3e = get_expression(adata, 'CD3E')
foxp3 = get_expression(adata, 'FOXP3')
cxcr5 = get_expression(adata, 'CXCR5')
ccr7 = get_expression(adata, 'CCR7')
lef1 = get_expression(adata, 'LEF1')

# Apply rules in order of specificity
# Treg: FOXP3+ (must check first before Tfh)
treg_mask = t_mask & (foxp3 > 0)
adata.obs.loc[treg_mask, 'refined_subtype'] = 'Treg'

# Tfr: CXCR5+ FOXP3+ (already captured above, but explicit)
tfr_mask = t_mask & (cxcr5 > 0) & (foxp3 > 0)
adata.obs.loc[tfr_mask, 'refined_subtype'] = 'Tfr'

# Tfh: CXCR5+ FOXP3- (check after Treg)
tfh_mask = t_mask & (cxcr5 > 0) & (foxp3 == 0)
adata.obs.loc[tfh_mask, 'refined_subtype'] = 'Tfh'

# T_CD8: CD8A+ or CD8B+
t_cd8_mask = t_mask & ((cd8a > 0) | (cd8b > 0)) & ~treg_mask & ~tfh_mask & ~tfr_mask
adata.obs.loc[t_cd8_mask, 'refined_subtype'] = 'T_CD8'

# T_helper: CD4+ CD8A-
t_helper_mask = t_mask & (cd4 > 0) & (cd8a == 0) & ~treg_mask & ~tfh_mask & ~tfr_mask
adata.obs.loc[t_helper_mask, 'refined_subtype'] = 'T_helper'

# T_naive: CCR7+ or LEF1+ (remaining T cells)
t_naive_mask = (t_mask & ((ccr7 > 0) | (lef1 > 0)) & 
                (adata.obs['refined_subtype'] == 'T_cell'))
adata.obs.loc[t_naive_mask, 'refined_subtype'] = 'T_naive'

# T_DN: remaining CD3E+ CD4- CD8A-
t_dn_mask = (t_mask & (cd3e > 0) & (cd4 == 0) & (cd8a == 0) & 
             (adata.obs['refined_subtype'] == 'T_cell'))
adata.obs.loc[t_dn_mask, 'refined_subtype'] = 'T_DN'

# Print T cell subtype distribution
t_subtypes = adata.obs.loc[t_mask, 'refined_subtype'].value_counts()
for st, n in t_subtypes.items():
    pct = 100 * n / t_mask.sum()
    print(f'  {st}: {n:,} ({pct:.1f}%)')

Refining subtypes within lineages...

T cells: 253,048


  T_helper: 83,247 (32.9%)
  T_DN: 62,055 (24.5%)
  T_naive: 40,904 (16.2%)
  T_CD8: 34,114 (13.5%)
  Treg: 12,341 (4.9%)
  Tfh: 11,586 (4.6%)
  T_cell: 7,772 (3.1%)
  Tfr: 1,029 (0.4%)


In [8]:
# --- B cell subtypes ---
b_mask = adata.obs['primary_lineage'] == 'B_cell'
print(f'\nB cells: {b_mask.sum():,}')

ms4a1 = get_expression(adata, 'MS4A1')
bcl6 = get_expression(adata, 'BCL6')
mki67 = get_expression(adata, 'MKI67')
top2a = get_expression(adata, 'TOP2A') if 'TOP2A' in adata.var_names else np.zeros(adata.n_obs)
xbp1 = get_expression(adata, 'XBP1')
mzb1 = get_expression(adata, 'MZB1') if 'MZB1' in adata.var_names else np.zeros(adata.n_obs)
cd27 = get_expression(adata, 'CD27')
fcer2 = get_expression(adata, 'FCER2') if 'FCER2' in adata.var_names else np.zeros(adata.n_obs)
tcl1a = get_expression(adata, 'TCL1A') if 'TCL1A' in adata.var_names else np.zeros(adata.n_obs)

# Plasma cells: XBP1 high, MS4A1 low
pc_mask = b_mask & (xbp1 > 0.5) & (ms4a1 < 0.3)
adata.obs.loc[pc_mask, 'refined_subtype'] = 'PC'

# Plasmablast: XBP1+, MS4A1+
pb_mask = b_mask & (xbp1 > 0) & (ms4a1 > 0) & ~pc_mask
adata.obs.loc[pb_mask, 'refined_subtype'] = 'PB'

# GC_DZ: Proliferating B cells (MKI67+ or TOP2A+)
# Note: BCL6 detection is low (~15%) due to marker dropout in Xenium
# Proliferation markers are more reliable for identifying dark zone cells
gc_dz_mask = b_mask & ((mki67 > 0) | (top2a > 0)) & ~pc_mask & ~pb_mask
adata.obs.loc[gc_dz_mask, 'refined_subtype'] = 'GC_DZ'

# GC_LZ: BCL6+ non-proliferating B cells
gc_lz_mask = b_mask & (bcl6 > 0) & (mki67 == 0) & (top2a == 0) & ~pc_mask & ~pb_mask & ~gc_dz_mask
adata.obs.loc[gc_lz_mask, 'refined_subtype'] = 'GC_LZ'

# NBC: FCER2+ or TCL1A+
nbc_mask = (b_mask & ((fcer2 > 0) | (tcl1a > 0)) &
            (adata.obs['refined_subtype'] == 'B_cell'))
adata.obs.loc[nbc_mask, 'refined_subtype'] = 'NBC'

# MBC: CD27+
mbc_mask = (b_mask & (cd27 > 0) &
            (adata.obs['refined_subtype'] == 'B_cell'))
adata.obs.loc[mbc_mask, 'refined_subtype'] = 'MBC'

# B_mature: remaining B cells
b_mature_mask = b_mask & (adata.obs['refined_subtype'] == 'B_cell')
adata.obs.loc[b_mature_mask, 'refined_subtype'] = 'B_mature'

# Print B cell subtype distribution
b_subtypes = adata.obs.loc[b_mask, 'refined_subtype'].value_counts()
for st, n in b_subtypes.items():
    pct = 100 * n / b_mask.sum()
    print(f'  {st}: {n:,} ({pct:.1f}%)')


B cells: 179,764


  B_mature: 76,847 (42.7%)
  NBC: 49,561 (27.6%)
  MBC: 20,505 (11.4%)
  PB: 19,263 (10.7%)
  GC_LZ: 5,901 (3.3%)
  PC: 5,404 (3.0%)
  GC_DZ: 2,283 (1.3%)


In [9]:
# --- Myeloid subtypes ---
mye_mask = adata.obs['primary_lineage'] == 'Myeloid'
print(f'\nMyeloid cells: {mye_mask.sum():,}')

cd68 = get_expression(adata, 'CD68')
cd14 = get_expression(adata, 'CD14')
mertk = get_expression(adata, 'MERTK') if 'MERTK' in adata.var_names else np.zeros(adata.n_obs)
il3ra = get_expression(adata, 'IL3RA') if 'IL3RA' in adata.var_names else np.zeros(adata.n_obs)
clec4c = get_expression(adata, 'CLEC4C') if 'CLEC4C' in adata.var_names else np.zeros(adata.n_obs)
cd1c = get_expression(adata, 'CD1C') if 'CD1C' in adata.var_names else np.zeros(adata.n_obs)
kit = get_expression(adata, 'KIT') if 'KIT' in adata.var_names else np.zeros(adata.n_obs)

# TBM: MERTK+
tbm_mask = mye_mask & (mertk > 0)
adata.obs.loc[tbm_mask, 'refined_subtype'] = 'TBM'

# pDC: IL3RA+ or CLEC4C+
pdc_mask = mye_mask & ((il3ra > 0) | (clec4c > 0)) & ~tbm_mask
adata.obs.loc[pdc_mask, 'refined_subtype'] = 'pDC'

# DC: CD1C+
dc_mask = mye_mask & (cd1c > 0) & ~tbm_mask & ~pdc_mask
adata.obs.loc[dc_mask, 'refined_subtype'] = 'DC'

# Mast: KIT+
mast_mask = mye_mask & (kit > 0) & ~tbm_mask & ~pdc_mask & ~dc_mask
adata.obs.loc[mast_mask, 'refined_subtype'] = 'Mast'

# Monocyte: CD14+
mono_mask = mye_mask & (cd14 > 0.3) & ~tbm_mask & ~pdc_mask & ~dc_mask & ~mast_mask
adata.obs.loc[mono_mask, 'refined_subtype'] = 'Monocyte'

# Macrophage: remaining myeloid
mac_mask = mye_mask & (adata.obs['refined_subtype'] == 'Myeloid')
adata.obs.loc[mac_mask, 'refined_subtype'] = 'Macrophage'

# --- Stromal subtypes ---
str_mask = adata.obs['primary_lineage'] == 'Stromal'
print(f'\nStromal cells: {str_mask.sum():,}')

pecam1 = get_expression(adata, 'PECAM1')
vwf = get_expression(adata, 'VWF') if 'VWF' in adata.var_names else np.zeros(adata.n_obs)
rgs5 = get_expression(adata, 'RGS5') if 'RGS5' in adata.var_names else np.zeros(adata.n_obs)
pdgfrb = get_expression(adata, 'PDGFRB') if 'PDGFRB' in adata.var_names else np.zeros(adata.n_obs)
cxcl12 = get_expression(adata, 'CXCL12')
ccl19 = get_expression(adata, 'CCL19') if 'CCL19' in adata.var_names else np.zeros(adata.n_obs)
cxcl13 = get_expression(adata, 'CXCL13')

# Endothelial: PECAM1+ or VWF+
endo_mask = str_mask & ((pecam1 > 0) | (vwf > 0))
adata.obs.loc[endo_mask, 'refined_subtype'] = 'Endothelial'

# Pericyte: RGS5+ or PDGFRB+
peri_mask = str_mask & ((rgs5 > 0) | (pdgfrb > 0)) & ~endo_mask
adata.obs.loc[peri_mask, 'refined_subtype'] = 'Pericyte'

# FDC: CXCL13+
fdc_mask = str_mask & (cxcl13 > 0) & ~endo_mask & ~peri_mask
adata.obs.loc[fdc_mask, 'refined_subtype'] = 'FDC'

# FRC: CXCL12+ or CCL19+
frc_mask = str_mask & ((cxcl12 > 0) | (ccl19 > 0)) & ~endo_mask & ~peri_mask & ~fdc_mask
adata.obs.loc[frc_mask, 'refined_subtype'] = 'FRC'

# Remaining stromal
str_other_mask = str_mask & (adata.obs['refined_subtype'] == 'Stromal')
adata.obs.loc[str_other_mask, 'refined_subtype'] = 'Stromal'

# --- FRC/FDC from Unassigned lineage ---
# Many FRC/FDC cells have low expression of canonical lineage markers
# but express chemokine markers (CXCL12 for FRC, CXCL13 for FDC)
unassigned_mask = adata.obs['primary_lineage'] == 'Unassigned'
print(f'\nUnassigned cells: {unassigned_mask.sum():,}')

# FDC from Unassigned: CXCL13+
fdc_from_unassigned = unassigned_mask & (cxcl13 > 0)
adata.obs.loc[fdc_from_unassigned, 'refined_subtype'] = 'FDC'
print(f'  -> FDC (CXCL13+): {fdc_from_unassigned.sum():,}')

# FRC from Unassigned: CXCL12+ (after excluding FDC)
frc_from_unassigned = unassigned_mask & (cxcl12 > 0) & ~fdc_from_unassigned
adata.obs.loc[frc_from_unassigned, 'refined_subtype'] = 'FRC'
print(f'  -> FRC (CXCL12+): {frc_from_unassigned.sum():,}')


Myeloid cells: 29,981



Stromal cells: 174,603



Unassigned cells: 62,091
  -> FDC (CXCL13+): 1,208
  -> FRC (CXCL12+): 0


In [10]:
# --- NK/NKT split ---
nk_mask = adata.obs['primary_lineage'] == 'NK'
print(f'\nNK lineage cells: {nk_mask.sum():,}')

cd3e = get_expression(adata, 'CD3E')
cd3d = get_expression(adata, 'CD3D')

# NK: CD3E- and CD3D-
nk_true = nk_mask & (cd3e == 0) & (cd3d == 0)
adata.obs.loc[nk_true, 'refined_subtype'] = 'NK'

# NKT: CD3E+ or CD3D+ (T cell receptor positive)
nkt_mask = nk_mask & ((cd3e > 0) | (cd3d > 0))
adata.obs.loc[nkt_mask, 'refined_subtype'] = 'NKT'

print(f'  NK (CD3E-): {nk_true.sum():,}')
print(f'  NKT (CD3E+): {nkt_mask.sum():,}')


NK lineage cells: 9,160


  NK (CD3E-): 3,415
  NKT (CD3E+): 5,745


---
## 4. Validation

We validate annotations using three criteria:

1. **Marker expression**: Check that expected markers are expressed
2. **Proportions**: Compare to expected proportions for lymph node
3. **Spatial co-localization**: Verify expected spatial relationships

In [11]:
# Final cell type distribution
print('Final cell type distribution:')
print('=' * 50)

ct_counts = adata.obs['refined_subtype'].value_counts()
for ct, n in ct_counts.items():
    pct = 100 * n / adata.n_obs
    print(f'{ct:<20}: {n:>8,} ({pct:>5.1f}%)')

print('=' * 50)
print(f'Total: {adata.n_obs:,}')
print(f'Assigned: {(adata.obs["refined_subtype"] != "Unassigned").sum():,}')

Final cell type distribution:
FRC                 :   90,630 ( 12.8%)
T_helper            :   83,247 ( 11.7%)
B_mature            :   76,847 ( 10.8%)
Endothelial         :   64,556 (  9.1%)
T_DN                :   62,055 (  8.8%)
Unassigned          :   60,883 (  8.6%)
NBC                 :   49,561 (  7.0%)
T_naive             :   40,904 (  5.8%)
T_CD8               :   34,114 (  4.8%)
MBC                 :   20,505 (  2.9%)
PB                  :   19,263 (  2.7%)
Pericyte            :   13,808 (  1.9%)
Treg                :   12,341 (  1.7%)
Macrophage          :   11,608 (  1.6%)
Tfh                 :   11,586 (  1.6%)
Monocyte            :    9,958 (  1.4%)
T_cell              :    7,772 (  1.1%)
GC_LZ               :    5,901 (  0.8%)
NKT                 :    5,745 (  0.8%)
PC                  :    5,404 (  0.8%)
FDC                 :    4,851 (  0.7%)
pDC                 :    4,285 (  0.6%)
NK                  :    3,415 (  0.5%)
TBM                 :    2,730 (  0.4%)
GC_DZ     

In [12]:
# Marker expression validation
MARKER_VALIDATION = {
    'T_naive': {'pos': ['CD3E', 'LEF1'], 'neg': ['MS4A1']},
    'T_helper': {'pos': ['CD3E', 'CD4'], 'neg': ['MS4A1', 'CD8A']},
    'T_CD8': {'pos': ['CD3E', 'CD8A'], 'neg': ['MS4A1']},
    'Treg': {'pos': ['CD3E', 'FOXP3'], 'neg': ['MS4A1']},
    'Tfh': {'pos': ['CD3E', 'CXCR5'], 'neg': ['MS4A1', 'FOXP3']},
    'GC_DZ': {'pos': ['MS4A1', 'MKI67', 'TOP2A'], 'neg': ['CD3E']},  # Proliferation markers, not BCL6
    'GC_LZ': {'pos': ['MS4A1', 'BCL6'], 'neg': ['CD3E', 'MKI67']},
    'NBC': {'pos': ['MS4A1'], 'neg': ['CD3E']},
    'MBC': {'pos': ['MS4A1', 'CD27'], 'neg': ['CD3E']},
    'PC': {'pos': ['XBP1'], 'neg': ['MS4A1']},
    'NK': {'pos': ['KLRB1'], 'neg': ['CD3E']},
    'NKT': {'pos': ['KLRB1', 'CD3E'], 'neg': []},
    'Macrophage': {'pos': ['CD68'], 'neg': ['MS4A1', 'CD3E']},
    'TBM': {'pos': ['CD68', 'MERTK'], 'neg': ['MS4A1']},
    'Endothelial': {'pos': ['PECAM1'], 'neg': ['MS4A1', 'CD3E']},
    'FRC': {'pos': ['CXCL12'], 'neg': ['MS4A1', 'CD3E']},
    'FDC': {'pos': ['CXCL13'], 'neg': ['CD3E']},
}

print('Marker validation:')
print(f'{"Cell Type":<15} {"n":>8}  {"Pos Markers":>12} {"Neg Markers":>12} {"Status":>8}')
print('-' * 65)

passed = 0
for ct, markers in MARKER_VALIDATION.items():
    mask = adata.obs['refined_subtype'] == ct
    n = mask.sum()
    if n == 0:
        continue

    # Check positive markers (any positive = pass)
    pos_fracs = [get_expr_fraction(adata, mask, m) for m in markers['pos'] if m in adata.var_names]
    max_pos = max(pos_fracs) * 100 if pos_fracs else 0

    # Check negative markers
    neg_fracs = [get_expr_fraction(adata, mask, m) for m in markers['neg'] if m in adata.var_names]
    avg_neg = np.mean(neg_fracs) * 100 if neg_fracs else 0

    # Status: pass if any pos > 30% and avg neg < 30%
    status = 'PASS' if max_pos > 30 and avg_neg < 30 else 'WARN'
    if status == 'PASS':
        passed += 1

    print(f'{ct:<15} {n:>8,}  {max_pos:>10.1f}%  {avg_neg:>10.1f}%  {status:>8}')

print('-' * 65)
print(f'Passed: {passed}/{len(MARKER_VALIDATION)}')

Marker validation:
Cell Type              n   Pos Markers  Neg Markers   Status
-----------------------------------------------------------------


T_naive           40,904        93.6%        12.5%      PASS


T_helper          83,247       100.0%         6.5%      PASS
T_CD8             34,114        93.7%        17.1%      PASS


Treg              12,341       100.0%        15.9%      PASS
Tfh               11,586       100.0%        13.1%      PASS
GC_DZ              2,283        91.1%        34.7%      WARN


GC_LZ              5,901       100.0%        14.8%      PASS
NBC               49,561        92.3%        32.6%      WARN


MBC               20,505       100.0%        40.9%      WARN
PC                 5,404       100.0%         0.0%      PASS
NK                 3,415        77.5%         0.0%      PASS
NKT                5,745       100.0%         0.0%      PASS


Macrophage        11,608        74.6%        13.7%      PASS
TBM                2,730       100.0%        12.3%      PASS


Endothelial       64,556       100.0%        21.0%      PASS


FRC               90,630        99.7%        24.6%      PASS
FDC                4,851       100.0%        27.5%      PASS
-----------------------------------------------------------------
Passed: 14/17


In [13]:
# Proportion validation
EXPECTED_PROPORTIONS = {
    'B cells': {'types': ['B_mature', 'NBC', 'MBC', 'GC_DZ', 'GC_LZ', 'PB', 'PC'], 
                'expected': (0.25, 0.55), 'note': 'Including all B subtypes'},
    'T cells': {'types': ['T_naive', 'T_helper', 'T_CD8', 'T_DN', 'Treg', 'Tfh', 'Tfr'],
                'expected': (0.25, 0.55), 'note': 'Including all T subtypes'},
    'NK+NKT': {'types': ['NK', 'NKT'], 'expected': (0.01, 0.10), 'note': 'NK lineage'},
    'Myeloid': {'types': ['Macrophage', 'TBM', 'Monocyte', 'DC', 'pDC', 'Myeloid', 'Mast'],
                'expected': (0.03, 0.15), 'note': 'All myeloid'},
    'Stromal': {'types': ['Endothelial', 'FRC', 'FDC', 'Pericyte', 'Stromal'],
                'expected': (0.05, 0.25), 'note': 'May be elevated in spatial data'},
}

n_assigned = (adata.obs['refined_subtype'] != 'Unassigned').sum()

print('Proportion validation:')
print(f'{"Lineage":<12} {"Count":>10} {"Observed":>10} {"Expected":>15} {"Status":>8}')
print('-' * 65)

for lineage, info in EXPECTED_PROPORTIONS.items():
    mask = adata.obs['refined_subtype'].isin(info['types'])
    n = mask.sum()
    pct = n / n_assigned
    exp_lo, exp_hi = info['expected']
    status = 'PASS' if exp_lo <= pct <= exp_hi else 'WARN'
    print(f'{lineage:<12} {n:>10,} {100*pct:>9.1f}% {100*exp_lo:>5.0f}-{100*exp_hi:<5.0f}% {status:>8}')

Proportion validation:
Lineage           Count   Observed        Expected   Status
-----------------------------------------------------------------
B cells         179,764      27.8%    25-55   %     PASS
T cells         245,276      37.9%    25-55   %     PASS
NK+NKT            9,160       1.4%     1-10   %     PASS
Myeloid          29,981       4.6%     3-15   %     PASS
Stromal         175,811      27.1%     5-25   %     WARN


In [14]:
# Spatial co-localization validation
print('Building spatial graph for co-localization validation...')

# Build k-NN spatial graph
coords = adata.obsm['spatial']
tree = cKDTree(coords)
_, indices = tree.query(coords, k=16)  # k=15 neighbors + self
indices = indices[:, 1:]  # Remove self

# Expected co-localizations
EXPECTED_COLOC = {
    ('GC_DZ', 'GC_LZ'): 'GC structure',
    ('GC_DZ', 'Tfh'): 'T-B interaction in GC',
    ('GC_LZ', 'Tfh'): 'T-B interaction in GC',
    ('GC_LZ', 'FDC'): 'GC organization',
    ('T_naive', 'T_helper'): 'T zone',
    ('T_naive', 'FRC'): 'T zone structure',
    ('NBC', 'MBC'): 'B follicle',
    ('Endothelial', 'Pericyte'): 'Vasculature',
}

cell_types = adata.obs['refined_subtype'].values
ct_to_idx = {ct: i for i, ct in enumerate(sorted(set(cell_types)))}

print('\nSpatial co-localization validation:')
print(f'{"Cell Type 1":<15} {"Cell Type 2":<15} {"Enrichment":>12} {"Biology":>25}')
print('-' * 75)

for (ct1, ct2), biology in EXPECTED_COLOC.items():
    if ct1 not in ct_to_idx or ct2 not in ct_to_idx:
        continue
    
    # Cells of type 1
    mask1 = cell_types == ct1
    n1 = mask1.sum()
    if n1 == 0:
        continue
    
    # Expected proportion (global)
    n2 = (cell_types == ct2).sum()
    expected_pct = n2 / len(cell_types)
    
    # Observed proportion in neighbors
    neighbor_types = cell_types[indices[mask1].flatten()]
    observed_pct = (neighbor_types == ct2).mean()
    
    # Enrichment
    enrichment = observed_pct / expected_pct if expected_pct > 0 else 0
    
    status = 'PASS' if enrichment > 1.5 else ('OK' if enrichment > 1.0 else 'WARN')
    print(f'{ct1:<15} {ct2:<15} {enrichment:>10.2f}x  {biology:>25}')

Building spatial graph for co-localization validation...



Spatial co-localization validation:
Cell Type 1     Cell Type 2       Enrichment                   Biology
---------------------------------------------------------------------------
GC_DZ           GC_LZ                 4.76x               GC structure
GC_DZ           Tfh                   1.66x      T-B interaction in GC
GC_LZ           Tfh                   1.07x      T-B interaction in GC
GC_LZ           FDC                   1.46x            GC organization
T_naive         T_helper              1.66x                     T zone
T_naive         FRC                   0.85x           T zone structure


NBC             MBC                   2.02x                 B follicle
Endothelial     Pericyte              1.32x                Vasculature


---
## 5. Finalize & Save

In [15]:
# Create final cell type column
adata.obs['cell_type'] = adata.obs['refined_subtype'].copy()

# Summary
print('=' * 60)
print('ANNOTATION SUMMARY')
print('=' * 60)
print(f'Total cells: {adata.n_obs:,}')

n_assigned = (adata.obs['cell_type'] != 'Unassigned').sum()
n_unassigned = (adata.obs['cell_type'] == 'Unassigned').sum()
print(f'Assigned: {n_assigned:,} ({100*n_assigned/adata.n_obs:.1f}%)')
print(f'Unassigned: {n_unassigned:,} ({100*n_unassigned/adata.n_obs:.1f}%)')

n_types = adata.obs['cell_type'].nunique()
print(f'Cell types: {n_types}')

# Show by lineage
print('\nBy lineage:')
for lineage in ['B_cell', 'T_cell', 'NK', 'Myeloid', 'Stromal']:
    mask = adata.obs['primary_lineage'] == lineage
    n = mask.sum()
    if n > 0:
        subtypes = adata.obs.loc[mask, 'cell_type'].nunique()
        print(f'  {lineage}: {n:,} cells, {subtypes} subtypes')

ANNOTATION SUMMARY
Total cells: 708,647
Assigned: 647,764 (91.4%)
Unassigned: 60,883 (8.6%)
Cell types: 29

By lineage:


  B_cell: 179,764 cells, 7 subtypes
  T_cell: 253,048 cells, 8 subtypes


  NK: 9,160 cells, 2 subtypes
  Myeloid: 29,981 cells, 6 subtypes
  Stromal: 174,603 cells, 5 subtypes


In [16]:
# Save annotated data
# Keep only essential columns
keep_cols = ['cell_type', 'primary_lineage', 'lineage_score_max']
keep_cols += [c for c in adata.obs.columns if c.endswith('_score') or c.endswith('_positive')]

# Create clean output
adata_clean = adata.copy()
adata_clean.obs = adata.obs[keep_cols]

# Save
output_path = '../data/xenium_lymphnode_annotated.h5ad'
adata_clean.write_h5ad(output_path)
print(f'Saved: {output_path}')
print(f'Shape: {adata_clean.shape}')
print(f'Columns: {list(adata_clean.obs.columns)}')

Saved: ../data/xenium_lymphnode_annotated.h5ad
Shape: (708647, 4624)
Columns: ['cell_type', 'primary_lineage', 'lineage_score_max', 'B_cell_score', 'B_cell_positive', 'T_cell_score', 'T_cell_positive', 'Myeloid_score', 'Myeloid_positive', 'Stromal_score', 'Stromal_positive', 'NK_score', 'NK_positive']


---
## Summary

This notebook demonstrated a **hierarchical cell type annotation workflow** for Xenium spatial transcriptomics:

1. **Lineage assignment** using canonical markers constrains cells to biologically valid lineages
2. **Subtype refinement** applies marker rules within each lineage
3. **Validation** confirms annotations using markers, proportions, and spatial co-localization

### Key Principles

- **GC_DZ cells**: Identified by proliferation markers (MKI67+, TOP2A+) rather than BCL6, which has low detection in spatial data (~15%)
- **FRC/FDC cells**: Many have low expression of canonical lineage markers and are recovered from "Unassigned" cells based on chemokine expression (CXCL12 for FRC, CXCL13 for FDC)
- **NKT vs NK**: Distinguished by CD3E expression (NKT is CD3E+)

### Key Advantages

- **No external dependencies**: Pure marker-based approach, no reference atlas or model required
- **Handles marker dropout**: Multiple redundant markers per lineage
- **Prevents cross-lineage contamination**: Subtype assignment constrained within lineages
- **Transparent & reproducible**: Deterministic marker rules rather than black-box models
- **Validated**: Quantitative checks against expected biology

### Using Pre-annotated Data

For downstream CONSTELLATION analysis, use the pre-annotated data:

```python
adata = sc.read_h5ad('../data/xenium_lymphnode.h5ad')
cell_type_col = 'cell_type'
```

### Output

The annotated data can be used for:
- Spatial ligand-receptor analysis with CONSTELLATION
- Cell type proportion comparisons
- Neighborhood composition analysis
- Tissue architecture characterization